# Model A — Fairness Pipeline: Clinical + Insurance States

This notebook extends the **Model B** (health-only) pipeline by incorporating **insurance status** as a non-clinical variable into the state definition.

The goal is to produce a second pair of transition matrices:
- **P_icu_A** — ICU dynamics under the augmented state space
- **P_nonicu_A** — non-ICU dynamics under the augmented state space

Comparing Model A vs Model B policies reveals whether insurance status influences the RMAB allocation decisions — i.e., whether the policy is biased by non-clinical factors.

### State Space Design

| Dimension | Variable | Levels |
|---|---|---|
| Clinical (unchanged) | MAP, SpO2, GCS, Lactate | 3⁴ = 81 live states |
| Non-clinical (new) | Insurance | 3 (Medicaid / Medicare / Other) |
| Death | — | 1 absorbing state |

$$\text{state\_A} = \text{insurance\_bin} \times 81 + \text{clinical\_state}$$

Total: **3 × 81 + 1 = 244 states**

| Insurance bin | Category | Rationale |
|---|---|---|
| 0 | Medicaid | Government — low income |
| 1 | Medicare | Government — elderly/disabled |
| 2 | Other | Private / self-pay / unspecified |

---
## Section 1 — Setup & Data Loading

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH = os.path.join(
    r'C:\Users\eric\Desktop\Mimic RMAB',
    'data', 'mimic-iv-demo',
    'mimic-iv-clinical-database-demo-2.2'
)
HOSP    = os.path.join(BASE_PATH, 'hosp')
ICU     = os.path.join(BASE_PATH, 'icu')
OUT_DIR = r'C:\Users\eric\Desktop\RMAB-MIMIC'

print('Base path:', BASE_PATH)
print('Exists:   ', os.path.isdir(BASE_PATH))
print('Output dir:', OUT_DIR)

In [ ]:
# ── Load tables ───────────────────────────────────────────────────────────────
patients    = pd.read_csv(os.path.join(HOSP, 'patients.csv.gz'), compression='gzip')
admissions  = pd.read_csv(os.path.join(HOSP, 'admissions.csv.gz'), compression='gzip',
                          parse_dates=['admittime', 'dischtime', 'deathtime',
                                       'edregtime', 'edouttime'],
                          low_memory=False)
labevents   = pd.read_csv(os.path.join(HOSP, 'labevents.csv.gz'), compression='gzip',
                          usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'],
                          parse_dates=['charttime'], low_memory=False)
icustays    = pd.read_csv(os.path.join(ICU, 'icustays.csv.gz'), compression='gzip',
                          parse_dates=['intime', 'outtime'])
chartevents = pd.read_csv(os.path.join(ICU, 'chartevents.csv.gz'), compression='gzip',
                          usecols=['subject_id', 'hadm_id', 'stay_id',
                                   'charttime', 'itemid', 'valuenum'],
                          parse_dates=['charttime'], low_memory=False)

for name, df in [('patients', patients), ('admissions', admissions),
                 ('labevents', labevents), ('icustays', icustays),
                 ('chartevents', chartevents)]:
    print(f'{name:<14} shape: {df.shape}')

---
## Section 2 — ICU / Non-ICU Day Labeling

*(Identical to Model B pipeline)*

In [ ]:
adm = admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime',
                   'hospital_expire_flag']].dropna(subset=['admittime', 'dischtime']).copy()
adm['admit_date'] = adm['admittime'].dt.normalize()
adm['disch_date'] = adm['dischtime'].dt.normalize()

rows = []
for _, row in adm.iterrows():
    for d in pd.date_range(row['admit_date'], row['disch_date'], freq='D'):
        rows.append({'subject_id': row['subject_id'], 'hadm_id': row['hadm_id'],
                     'date': d, 'hospital_expire_flag': row['hospital_expire_flag']})
daily = pd.DataFrame(rows)

icu_rows = []
for _, stay in icustays.iterrows():
    for d in pd.date_range(stay['intime'].normalize(), stay['outtime'].normalize(), freq='D'):
        icu_rows.append({'subject_id': stay['subject_id'], 'hadm_id': stay['hadm_id'], 'date': d})
icu_days = pd.DataFrame(icu_rows).drop_duplicates(subset=['subject_id', 'hadm_id', 'date'])
icu_days['is_icu'] = 1

daily = daily.merge(icu_days[['subject_id', 'hadm_id', 'date', 'is_icu']],
                    on=['subject_id', 'hadm_id', 'date'], how='left')
daily['is_icu'] = daily['is_icu'].fillna(0).astype(int)

print(f'Daily rows: {len(daily):,}  |  ICU days: {daily["is_icu"].sum():,}')

---
## Section 3 — Vitals & Labs Extraction

*(Identical to Model B pipeline)*

In [ ]:
# ── Vitals ────────────────────────────────────────────────────────────────────
VITAL_ITEMS = {220045: 'heart_rate', 220181: 'map', 220210: 'resp_rate',
               220277: 'spo2', 223762: 'temp_c', 223761: 'temp_f'}
GCS_ITEMS   = [220739, 223900, 223901]

ce_vitals = chartevents[chartevents['itemid'].isin(list(VITAL_ITEMS) + GCS_ITEMS)].copy()
ce_vitals['date'] = ce_vitals['charttime'].dt.normalize()

# Non-GCS: daily mean
ce_nonGCS = ce_vitals[ce_vitals['itemid'].isin(VITAL_ITEMS)].copy()
ce_nonGCS['vital'] = ce_nonGCS['itemid'].map(VITAL_ITEMS)
vitals_daily = (ce_nonGCS.groupby(['subject_id', 'date', 'vital'])['valuenum'].mean()
                .reset_index()
                .pivot_table(index=['subject_id', 'date'], columns='vital',
                             values='valuenum', aggfunc='mean')
                .reset_index())
vitals_daily.columns.name = None

if 'temp_f' in vitals_daily.columns and 'temp_c' in vitals_daily.columns:
    vitals_daily['temperature'] = vitals_daily['temp_c'].combine_first(
        (vitals_daily['temp_f'] - 32) / 1.8)
    vitals_daily.drop(columns=['temp_c', 'temp_f'], inplace=True)
elif 'temp_c' in vitals_daily.columns:
    vitals_daily.rename(columns={'temp_c': 'temperature'}, inplace=True)
elif 'temp_f' in vitals_daily.columns:
    vitals_daily['temperature'] = (vitals_daily['temp_f'] - 32) / 1.8
    vitals_daily.drop(columns=['temp_f'], inplace=True)

# GCS: sum sub-scores per moment, then daily mean
ce_gcs = ce_vitals[ce_vitals['itemid'].isin(GCS_ITEMS)].copy()
gcs_moment = (ce_gcs.groupby(['subject_id', 'stay_id', 'charttime'])['valuenum']
              .sum().reset_index().rename(columns={'valuenum': 'gcs_total'}))
gcs_moment = gcs_moment[(gcs_moment['gcs_total'] >= 3) & (gcs_moment['gcs_total'] <= 15)]
gcs_moment['date'] = gcs_moment['charttime'].dt.normalize()
gcs_daily = gcs_moment.groupby(['subject_id', 'date'])['gcs_total'].mean().reset_index()

vitals_daily = vitals_daily.merge(gcs_daily, on=['subject_id', 'date'], how='outer')
print('vitals_daily shape:', vitals_daily.shape)

In [ ]:
# ── Labs ──────────────────────────────────────────────────────────────────────
LAB_ITEMS = {50813: 'lactate', 50912: 'creatinine'}
le = labevents[labevents['itemid'].isin(LAB_ITEMS)].copy()
le['lab']  = le['itemid'].map(LAB_ITEMS)
le['date'] = le['charttime'].dt.normalize()

labs_daily_raw = (le.groupby(['subject_id', 'date', 'lab'])['valuenum'].mean()
                  .reset_index()
                  .pivot_table(index=['subject_id', 'date'], columns='lab',
                               values='valuenum', aggfunc='mean')
                  .reset_index())
labs_daily_raw.columns.name = None

patient_dates = daily[['subject_id', 'date']].drop_duplicates()
labs_daily = patient_dates.merge(labs_daily_raw, on=['subject_id', 'date'], how='left')
labs_daily = labs_daily.sort_values(['subject_id', 'date'])
labs_daily[['lactate', 'creatinine']] = (
    labs_daily.groupby('subject_id')[['lactate', 'creatinine']].ffill())
print('labs_daily shape:', labs_daily.shape)

---
## Section 4 — Feature Assembly & Imputation

In [ ]:
features = daily[['subject_id', 'hadm_id', 'date', 'is_icu', 'hospital_expire_flag']].copy()
features = features.merge(vitals_daily, on=['subject_id', 'date'], how='left')
features = features.merge(labs_daily,   on=['subject_id', 'date'], how='left')

feature_cols = ['map', 'spo2', 'gcs_total', 'lactate',
                'heart_rate', 'resp_rate', 'temperature', 'creatinine']

# Forward-fill then median impute
features = features.sort_values(['subject_id', 'date'])
features[feature_cols] = features.groupby('subject_id')[feature_cols].ffill()
features[feature_cols] = features[feature_cols].fillna(features[feature_cols].median())

print('features shape:', features.shape)
print('Remaining NaNs:', features[feature_cols].isna().sum().sum())

---
## Section 5 — Insurance Extraction (Model A — NEW)

We pull the `insurance` field from the `admissions` table and map it to 3 bins:

| Bin | Category | Original value |
|---|---|---|
| 0 | Medicaid | `Medicaid` |
| 1 | Medicare | `Medicare` |
| 2 | Other | `Other` (private / self-pay / unspecified) |

The insurance bin is constant for an entire admission (it does not change day-to-day).

In [ ]:
# ── Insurance distribution ────────────────────────────────────────────────────
print('Insurance value counts in admissions:')
print(admissions['insurance'].value_counts())
print(f'\nMissing: {admissions["insurance"].isna().sum()}')

In [ ]:
# ── Map insurance to bin ──────────────────────────────────────────────────────
INS_MAP = {
    'Medicaid': 0,
    'Medicare': 1,
    'Other':    2,
}

ins = admissions[['hadm_id', 'insurance']].copy()
ins['insurance_bin'] = ins['insurance'].map(INS_MAP).fillna(2).astype(int)

print('Insurance bin distribution:')
print(ins['insurance_bin'].value_counts().sort_index()
      .rename({0: 'Medicaid', 1: 'Medicare', 2: 'Other'}))

# Merge onto features (one value per admission)
features = features.merge(ins[['hadm_id', 'insurance_bin']], on='hadm_id', how='left')
features['insurance_bin'] = features['insurance_bin'].fillna(2).astype(int)

print(f'\nRows with insurance info: {features["insurance_bin"].notna().sum():,} / {len(features):,}')

---
## Section 6 — State Discretization

Clinical bins are identical to Model B. The composite state index is then augmented with the insurance bin.

In [ ]:
# ── Clinical bins (same as Model B) ──────────────────────────────────────────
features['map_bin'] = pd.cut(
    features['map'], bins=[-np.inf, 65, 100, np.inf], labels=[0, 1, 2]).astype(int)
features['spo2_bin'] = pd.cut(
    features['spo2'], bins=[-np.inf, 90, 95, np.inf], labels=[0, 1, 2]).astype(int)
features['gcs_bin'] = pd.cut(
    features['gcs_total'], bins=[2, 8, 12, 15], labels=[0, 1, 2]).astype(int)
features['lactate_bin'] = pd.cut(
    features['lactate'], bins=[-np.inf, 2, 4, np.inf], labels=[2, 1, 0]).astype(int)

# ── Clinical state (0-80) ─────────────────────────────────────────────────────
N_CLINICAL   = 81   # 3^4
N_INS_BINS   = 3
N_STATES_A   = N_CLINICAL * N_INS_BINS + 1   # 244
DEATH_STATE  = N_STATES_A - 1                 # 243

features['clinical_state'] = (
    features['map_bin']     * 27 +
    features['spo2_bin']    *  9 +
    features['gcs_bin']     *  3 +
    features['lactate_bin']
)

# ── Augmented state (Model A) ─────────────────────────────────────────────────
# state_A = insurance_bin * 81 + clinical_state
features['state_A'] = features['insurance_bin'] * N_CLINICAL + features['clinical_state']

# ── Death state ────────────────────────────────────────────────────────────────
last_day  = features.groupby('hadm_id')['date'].transform('max')
died_mask = (features['hospital_expire_flag'] == 1) & (features['date'] == last_day)
features.loc[died_mask, 'state_A'] = DEATH_STATE

print(f'N_STATES_A = {N_STATES_A}  (DEATH_STATE = {DEATH_STATE})')
print(f'State_A range: {features["state_A"].min()} – {features["state_A"].max()}')
print(f'\nPatients who died: {features[died_mask]["subject_id"].nunique()}')
print(f'Unique states observed: {features["state_A"].nunique()} / {N_STATES_A}')

In [ ]:
# ── State distribution by insurance group ─────────────────────────────────────
ins_labels = {0: 'Medicaid', 1: 'Medicare', 2: 'Other'}
for b in range(N_INS_BINS):
    mask = (features['insurance_bin'] == b) & (features['state_A'] != DEATH_STATE)
    n_days   = mask.sum()
    n_states = features.loc[mask, 'state_A'].nunique()
    print(f'Insurance bin {b} ({ins_labels[b]}): {n_days:,} patient-days, '
          f'{n_states} unique states')

---
## Section 7 — Transition Matrix Estimation (Model A)

The same consecutive-day pair logic as Model B, but using `state_A` instead of `state`.

In [ ]:
# ── Build transition pairs ────────────────────────────────────────────────────
feat_sorted = features.sort_values(['subject_id', 'hadm_id', 'date']).reset_index(drop=True)
feat_sorted['state_A_next'] = (
    feat_sorted.groupby(['subject_id', 'hadm_id'])['state_A'].shift(-1))

transitions = feat_sorted.dropna(subset=['state_A_next']).copy()
transitions['state_A']      = transitions['state_A'].astype(int)
transitions['state_A_next'] = transitions['state_A_next'].astype(int)

print(f'Total transition pairs: {len(transitions):,}')
print(f'  ICU:     {transitions["is_icu"].sum():,}')
print(f'  Non-ICU: {(1 - transitions["is_icu"]).sum():,}')
print()

# Breakdown by insurance group
for b in range(N_INS_BINS):
    n = (transitions['insurance_bin'] == b).sum()
    print(f'  {ins_labels[b]}: {n:,} transitions')

In [ ]:
# ── Count matrices ────────────────────────────────────────────────────────────
count_icu_A    = np.zeros((N_STATES_A, N_STATES_A), dtype=np.float64)
count_nonicu_A = np.zeros((N_STATES_A, N_STATES_A), dtype=np.float64)

for _, row in transitions.iterrows():
    s, s_next, icu = int(row['state_A']), int(row['state_A_next']), int(row['is_icu'])
    if icu == 1:
        count_icu_A[s, s_next]    += 1
    else:
        count_nonicu_A[s, s_next] += 1

print(f'ICU    total counts: {count_icu_A.sum():.0f}')
print(f'NonICU total counts: {count_nonicu_A.sum():.0f}')

In [ ]:
# ── Row-normalise ─────────────────────────────────────────────────────────────
def normalise_rows(count_mat, n_states):
    prob = count_mat.copy()
    row_sums = prob.sum(axis=1, keepdims=True)
    nonzero  = (row_sums.squeeze() > 0)
    prob[nonzero]  = prob[nonzero] / row_sums[nonzero]
    prob[~nonzero] = 1.0 / n_states
    return prob, (~nonzero).sum()

P_icu_A,    n_zero_icu    = normalise_rows(count_icu_A,    N_STATES_A)
P_nonicu_A, n_zero_nonicu = normalise_rows(count_nonicu_A, N_STATES_A)

# Death is absorbing
P_icu_A[DEATH_STATE, :]              = 0.0
P_icu_A[DEATH_STATE, DEATH_STATE]    = 1.0
P_nonicu_A[DEATH_STATE, :]           = 0.0
P_nonicu_A[DEATH_STATE, DEATH_STATE] = 1.0

print(f'P_icu_A    zero rows (uniform fallback): {n_zero_icu}')
print(f'P_nonicu_A zero rows (uniform fallback): {n_zero_nonicu}')
print(f'Row-sum check P_icu_A:    min={P_icu_A.sum(1).min():.4f}  max={P_icu_A.sum(1).max():.4f}')
print(f'Row-sum check P_nonicu_A: min={P_nonicu_A.sum(1).min():.4f}  max={P_nonicu_A.sum(1).max():.4f}')

In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
np.save(os.path.join(OUT_DIR, 'P_icu_A.npy'),    P_icu_A)
np.save(os.path.join(OUT_DIR, 'P_nonicu_A.npy'), P_nonicu_A)
print(f'Saved P_icu_A.npy    shape={P_icu_A.shape}')
print(f'Saved P_nonicu_A.npy shape={P_nonicu_A.shape}')

---
## Section 8 — Visualization

Three plots:
1. **Heatmaps** of P_icu_A and P_nonicu_A (observed states only, log scale)
2. **Insurance group self-loop comparison** — for the top observed clinical states, compare the self-loop probability across insurance bins under ICU vs non-ICU
3. **Cross-insurance transition rates** — probability of transitioning to a state in a *different* insurance group (should be ~0 if insurance is fixed per admission; confirms correctness)

In [ ]:
# ── Observed states ───────────────────────────────────────────────────────────
obs_A = np.where((count_icu_A.sum(axis=1) + count_nonicu_A.sum(axis=1)) > 0)[0]
print(f'States with ≥1 transition: {len(obs_A)} / {N_STATES_A}')

# Annotate each observed state with its insurance group and clinical state
def decode_state_A(s):
    if s == DEATH_STATE:
        return 'Death', -1
    ins_b = s // N_CLINICAL
    clin  = s % N_CLINICAL
    return ins_labels[ins_b], clin

for s in obs_A[:10]:
    ins_name, clin = decode_state_A(s)
    print(f'  state {s:3d}  →  insurance={ins_name}, clinical_state={clin}')

In [ ]:
# ── Plot 1: P_icu_A heatmap ───────────────────────────────────────────────────
P_icu_A_obs  = P_icu_A[np.ix_(obs_A, obs_A)]
data_log     = np.log10(np.where(P_icu_A_obs > 0, P_icu_A_obs, 1e-6))
tick_step    = max(1, len(obs_A) // 12)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(data_log, aspect='auto', cmap='YlOrRd', vmin=-6, vmax=0)
ax.set_title(r'P_icu_A — Transition Matrix (ICU, Model A: clinical + insurance, $\log_{10}$ scale)'
             + f'\nShowing {len(obs_A)} observed states', fontsize=12)
ax.set_xlabel('Next-day state index')
ax.set_ylabel('Current-day state index')
ax.set_xticks(range(0, len(obs_A), tick_step))
ax.set_xticklabels(obs_A[::tick_step], rotation=45, fontsize=7)
ax.set_yticks(range(0, len(obs_A), tick_step))
ax.set_yticklabels(obs_A[::tick_step], fontsize=7)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label(r'$\log_{10}$(probability)')

# Shade regions by insurance group
for b in range(N_INS_BINS):
    states_in_group = [i for i, s in enumerate(obs_A)
                       if s != DEATH_STATE and s // N_CLINICAL == b]
    if states_in_group:
        lo, hi = states_in_group[0] - 0.5, states_in_group[-1] + 0.5
        for spine_ax in [ax]:
            spine_ax.axhline(lo, color='navy', lw=0.4, alpha=0.5)
            spine_ax.axhline(hi, color='navy', lw=0.4, alpha=0.5)
            spine_ax.axvline(lo, color='navy', lw=0.4, alpha=0.5)
            spine_ax.axvline(hi, color='navy', lw=0.4, alpha=0.5)
        mid = (lo + hi) / 2
        ax.text(-2.5, mid, ins_labels[b], fontsize=7, va='center',
                ha='right', color='navy', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'heatmap_P_icu_A.png'), dpi=120)
plt.show()
print('Saved heatmap_P_icu_A.png')

In [ ]:
# ── Plot 2: P_nonicu_A heatmap ────────────────────────────────────────────────
P_nonicu_A_obs = P_nonicu_A[np.ix_(obs_A, obs_A)]
data_log2      = np.log10(np.where(P_nonicu_A_obs > 0, P_nonicu_A_obs, 1e-6))

fig, ax = plt.subplots(figsize=(11, 9))
im2 = ax.imshow(data_log2, aspect='auto', cmap='YlOrRd', vmin=-6, vmax=0)
ax.set_title(r'P_nonicu_A — Transition Matrix (Non-ICU, Model A: clinical + insurance, $\log_{10}$ scale)'
             + f'\nShowing {len(obs_A)} observed states', fontsize=12)
ax.set_xlabel('Next-day state index')
ax.set_ylabel('Current-day state index')
ax.set_xticks(range(0, len(obs_A), tick_step))
ax.set_xticklabels(obs_A[::tick_step], rotation=45, fontsize=7)
ax.set_yticks(range(0, len(obs_A), tick_step))
ax.set_yticklabels(obs_A[::tick_step], fontsize=7)
cb2 = fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)
cb2.set_label(r'$\log_{10}$(probability)')

for b in range(N_INS_BINS):
    states_in_group = [i for i, s in enumerate(obs_A)
                       if s != DEATH_STATE and s // N_CLINICAL == b]
    if states_in_group:
        lo, hi = states_in_group[0] - 0.5, states_in_group[-1] + 0.5
        ax.axhline(lo, color='navy', lw=0.4, alpha=0.5)
        ax.axhline(hi, color='navy', lw=0.4, alpha=0.5)
        ax.axvline(lo, color='navy', lw=0.4, alpha=0.5)
        ax.axvline(hi, color='navy', lw=0.4, alpha=0.5)
        ax.text(-2.5, (lo + hi) / 2, ins_labels[b], fontsize=7, va='center',
                ha='right', color='navy', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'heatmap_P_nonicu_A.png'), dpi=120)
plt.show()
print('Saved heatmap_P_nonicu_A.png')

In [ ]:
# ── Plot 3: Self-loop probability by insurance group ──────────────────────────
# For the top-5 most visited clinical states (excluding death),
# compare P[s,s] across insurance bins × ICU/non-ICU

top_clinical = (features[features['state_A'] != DEATH_STATE]['clinical_state']
                .value_counts().head(5).index.tolist())

fig, axes = plt.subplots(1, len(top_clinical), figsize=(14, 5), sharey=True)
colors_ins = ['#e74c3c', '#3498db', '#2ecc71']

for ax, clin in zip(axes, top_clinical):
    x = np.arange(N_INS_BINS)
    icu_vals    = []
    nonicu_vals = []
    for b in range(N_INS_BINS):
        s = b * N_CLINICAL + clin
        icu_vals.append(P_icu_A[s, s])
        nonicu_vals.append(P_nonicu_A[s, s])
    width = 0.35
    bars1 = ax.bar(x - width/2, icu_vals,    width, color='steelblue', alpha=0.8, label='ICU')
    bars2 = ax.bar(x + width/2, nonicu_vals, width, color='darkorange', alpha=0.8, label='Non-ICU')
    ax.set_title(f'Clinical state {clin}', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([ins_labels[b] for b in range(N_INS_BINS)], fontsize=8, rotation=15)
    ax.set_ylim(0, 1.1)
    for bar in list(bars1) + list(bars2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

axes[0].set_ylabel('Self-loop probability P[s, s]')
axes[-1].legend(loc='upper right', fontsize=8)
fig.suptitle('Self-loop probability by insurance group — ICU vs Non-ICU\n'
             '(Top 5 most visited clinical states)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'selfloop_by_insurance.png'), dpi=120)
plt.show()
print('Saved selfloop_by_insurance.png')

---
## Section 9 — Summary

In [ ]:
print('=' * 55)
print('       MODEL A PIPELINE SUMMARY')
print('=' * 55)
print(f'  State space size (N_STATES_A)     : {N_STATES_A}')
print(f'  Clinical states per ins. group    : {N_CLINICAL}')
print(f'  Insurance groups                  : {N_INS_BINS} (Medicaid / Medicare / Other)')
print(f'  Death state index                 : {DEATH_STATE}')
print(f'  Unique states observed            : {len(obs_A)}')
print(f'  Total transition pairs            : {len(transitions):,}')
print(f'    ICU                             : {int(transitions["is_icu"].sum()):,}')
print(f'    Non-ICU                         : {int((1-transitions["is_icu"]).sum()):,}')
print(f'  P_icu_A zero rows (uniform)       : {n_zero_icu}')
print(f'  P_nonicu_A zero rows (uniform)    : {n_zero_nonicu}')
print('=' * 55)
print('\nOutput files:')
print(f'  P_icu_A.npy    : {P_icu_A.shape}')
print(f'  P_nonicu_A.npy : {P_nonicu_A.shape}')
print(f'  heatmap_P_icu_A.png')
print(f'  heatmap_P_nonicu_A.png')
print(f'  selfloop_by_insurance.png')